# Week 2 — Data Visualization & Exploratory Data Analysis
## Invoice Payment Delay & Cash-Flow Intelligence Platform

This notebook explores the Week 1 cleaned dataset. It separates paid invoices from open invoices because realized payment duration and delay are not available for open invoices.

**Input:** `../data/processed/cleaned_invoices.csv`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = '../data/processed/cleaned_invoices.csv'
df = pd.read_csv(DATA_PATH)
date_cols = ['payment_date','invoice_date','document_create_date','document_create_date_posted','due_date','baseline_create_date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')
paid = df[df['payment_date'].notna()].copy()
open_invoices = df[df['payment_date'].isna()].copy()
print('Shape:', df.shape, '| Paid:', len(paid), '| Open:', len(open_invoices))
display(df.head())


## 1. Data quality and summary statistics


In [ ]:
display(df.dtypes.to_frame('dtype'))
display(df.isna().sum().sort_values(ascending=False).to_frame('missing'))
flag_cols = ['invalid_amount_flag','due_before_invoice_flag','status_date_conflict_flag','payment_before_invoice_flag']
display(df[flag_cols].sum().to_frame('flagged_rows'))
display(df[['amount','days_to_pay','delay_days']].describe(percentiles=[.25,.5,.75,.9,.95,.99]))


## 2. Univariate analysis


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
df['is_open'].value_counts().sort_index().rename(index={0:'Paid',1:'Open'}).plot(kind='bar', ax=ax[0])
ax[0].set_title('Invoice status'); ax[0].set_ylabel('Invoices')
ax[1].hist(df['amount'], bins=50); ax[1].set_title('Invoice amount distribution')
ax[1].set_xlabel('Amount (currencies should be analyzed separately)')
plt.tight_layout()
plt.show()
paid['delay_bucket'] = pd.cut(paid['delay_days'], [-np.inf,-1,0,7,30,np.inf], labels=['Early','On time','1–7 days late','8–30 days late','>30 days late'])
paid['delay_bucket'].value_counts().plot(kind='bar', title='Paid invoices by delay category', figsize=(8,4))
plt.ylabel('Invoices'); plt.tight_layout(); plt.show()


## 3. Bivariate analysis


In [ ]:
fig, ax = plt.subplots(1,2,figsize=(12,4))
ax[0].scatter(paid['amount'], paid['delay_days'], alpha=.12, s=8)
ax[0].set_xlabel('Invoice amount'); ax[0].set_ylabel('Delay days')
corr = paid[['amount','days_to_pay','delay_days']].corr()
im=ax[1].imshow(corr, vmin=-1, vmax=1); ax[1].set_xticks(range(3),corr.columns,rotation=30); ax[1].set_yticks(range(3),corr.index)
fig.colorbar(im,ax=ax[1]); plt.tight_layout(); plt.show()


## 4. Time-series analysis


In [ ]:
paid['payment_month'] = paid['payment_date'].dt.to_period('M').astype(str)
monthly = paid.groupby('payment_month').agg(paid_invoices=('invoice_id','count'), late_rate=('paid_late_flag','mean'), avg_delay=('delay_days','mean')).reset_index()
fig, ax = plt.subplots(2,1,figsize=(12,7),sharex=True)
ax[0].plot(monthly.payment_month,monthly.paid_invoices,marker='o'); ax[0].set_title('Paid invoice volume')
ax[1].plot(monthly.payment_month,monthly.late_rate*100,marker='o'); ax[1].set_title('Late-payment rate'); ax[1].set_ylabel('% late')
plt.xticks(rotation=45); plt.tight_layout(); plt.show(); display(monthly)


## 5. Customer, payment-term and business analysis


In [ ]:
customer = paid.groupby('customer_id').agg(invoices=('invoice_id','count'),late_rate=('paid_late_flag','mean'),avg_delay=('delay_days','mean'),amount=('amount','sum')).sort_values('invoices',ascending=False)
display(customer.head(15))
terms = paid.groupby('cust_payment_terms').agg(invoices=('invoice_id','count'),late_rate=('paid_late_flag','mean'),avg_delay=('delay_days','mean'))
display(terms[terms.invoices >= 100].sort_values('late_rate',ascending=False))
business = paid.groupby('business_code').agg(invoices=('invoice_id','count'),late_rate=('paid_late_flag','mean'),avg_delay=('delay_days','mean')).sort_values('invoices',ascending=False)
display(business)


## 6. Open-invoice aging and cash-flow exposure

Open invoices have no payment date, so do not assign them a realized delay of zero. Report USD and CAD separately; do not add unlike currencies without FX conversion.


In [ ]:
display(open_invoices.groupby('currency').agg(open_invoices=('invoice_id','count'),open_amount=('amount','sum'),earliest_due=('due_date','min'),latest_due=('due_date','max')))
AS_OF_DATE = pd.Timestamp('2020-05-22') # latest observed payment date in this dataset
open_invoices['days_past_due_as_of'] = (AS_OF_DATE - pd.to_datetime(open_invoices['due_date'])).dt.days
open_invoices['aging_bucket'] = pd.cut(open_invoices['days_past_due_as_of'],[-np.inf,0,30,60,90,np.inf],labels=['Not yet due','1–30 days','31–60 days','61–90 days','>90 days'])
display(open_invoices.groupby(['currency','aging_bucket'],observed=False).agg(invoices=('invoice_id','count'),amount=('amount','sum')))


## 7. Initial business insights

- Late-payment rate is calculated only among paid invoices; open invoices are excluded.
- Use `customer_id` rather than `customer_name` for customer grouping because names contain text variants.
- Report USD and CAD separately for financial totals.
- Review records where `due_before_invoice_flag` is True.
- Use a chronological train/test split for later prediction work.
- Exclude post-payment fields such as `payment_date`, `days_to_pay`, and `delay_days` from predictors to prevent target leakage.
